# Egresados de educación superior por campo de estudio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/profesiones_pais/blob/main/notebooks/01_descarga_y_panel.ipynb)

Pipeline end-to-end del proyecto **profesiones_pais**: descarga de graduados
por campo ISCED-F 2013 a nivel *narrow* (F011, F021, ...) desde Eurostat,
crosswalk SPU → ISCED-F para Argentina, y consolidación del panel
`iso3 × year × isced_level × iscedf_narrow`.

**Notas sobre las fuentes (verificado contra la API el 2026-07-22):**
- `educ_uoe_grad02` viene en `unit=NR` (conteos absolutos) → fuente primaria.
- `educ_uoe_grad10` viene solo en `unit=PC` y es la **distribución por sexo
  dentro de cada campo** (no la composición por campo), por eso no se usa
  para reconstruir absolutos.

In [ ]:
# Setup: instala dependencias y clona el repo si estamos en Colab
import os, pathlib, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "requests", "pyarrow", "matplotlib", "openpyxl"], check=True)
    if not pathlib.Path("/content/profesiones_pais").exists():
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/santiagoriverti/profesiones_pais.git",
                        "/content/profesiones_pais"], check=True)
    os.chdir("/content/profesiones_pais")
else:
    root = pathlib.Path.cwd()
    if not (root / "src").exists():
        root = root.parent  # el notebook vive en notebooks/
    os.chdir(root)

sys.path.insert(0, str(pathlib.Path("src").resolve()))
print("Directorio de trabajo:", os.getcwd())

## Paso 1 — Descarga desde Eurostat

`fetch_graduates()` baja `educ_uoe_grad02` (todos los países y campos,
ED6/ED7/ED8, sexo total, 2013 en adelante) y cachea el crudo en
`data/raw/` con timestamp: las corridas siguientes no re-descargan.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

from eurostat_api import fetch_graduates, is_narrow

df = fetch_graduates()
print(f"{len(df):,} filas | {df['geo'].nunique()} geografías | "
      f"años {df['year'].min()}-{df['year'].max()} | "
      f"{int(df['iscedf13'].map(is_narrow).sum()):,} filas a nivel narrow")
df.head()

## Paso 2 — Panel consolidado y cobertura

Se filtran los campos *narrow* (`F` + 3 dígitos), se convierten los códigos
geo de Eurostat a ISO3 y se escribe `data/processed/panel.parquet`. El
reporte de cobertura (`coverage.csv`) distingue países con datos a nivel
narrow de los que solo reportan a nivel broad — eso define la muestra.

In [ ]:
from build_panel import main as build_panel_main

panel = build_panel_main()
panel.head()

## Paso 3 — Crosswalk SPU → ISCED-F (Argentina)

Argentina no reporta a Eurostat: sus egresados vienen por las disciplinas
de la SPU (`data/external/profesiones_arg.xlsx`, Síntesis de Información
Universitaria, 2014-2023) y `build_panel` los integra automáticamente vía el
crosswalk `data/reference/spu_to_iscedf_narrow.csv` (pensado para revisión
manual). Mapeo de niveles: Grado → ED6; Maestría y Especialidad → ED7;
Doctorado → ED8 (Pregrado y "Posgrado/Otros" quedan fuera). Abajo se listan
los casos del crosswalk que requieren decisión.

In [ ]:
import pandas as pd
from crosswalk import load_crosswalk

pd.set_option("display.max_colwidth", None)
cw = load_crosswalk()
print("Disciplinas mapeadas:", len(cw))
print(cw["confianza"].value_counts().to_string())
cw[cw["confianza"] != "alta"][["spu_disciplina", "iscedf_narrow", "confianza", "nota"]]

## Gráficos exploratorios

In [ ]:
import matplotlib.pyplot as plt

# Paleta categórica en orden fijo (validada para visión de color)
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

BROAD_LABELS = {
    "F00": "Genéricos", "F01": "Educación", "F02": "Artes y humanidades",
    "F03": "Cs. sociales y periodismo", "F04": "Negocios, adm. y derecho",
    "F05": "Cs. naturales y matemática", "F06": "TIC",
    "F07": "Ingeniería y construcción", "F08": "Agro y veterinaria",
    "F09": "Salud y bienestar", "F10": "Servicios",
}

# Composición de egresados de grado (ED6) por campo broad, último año común
paises = ["ARG", "DEU", "ESP", "FRA", "ITA", "POL", "SWE"]
ed6 = panel[(panel["isced_level"] == "ED6") & panel["iso3"].isin(paises)]
anio = int(ed6.groupby("iso3")["year"].max().min())  # último año con datos en todos

comp = ed6[ed6["year"] == anio].copy()
comp["broad"] = comp["iscedf_narrow"].str[:3]
comp = comp.groupby(["iso3", "broad"])["graduates"].sum().unstack(fill_value=0)
shares = comp.div(comp.sum(axis=1), axis=0) * 100

# Top 7 campos + "Otros" para mantener ≤ 8 categorías
top = shares.mean().nlargest(7).index.tolist()
plot_df = shares[top].rename(columns=BROAD_LABELS)
plot_df["Otros"] = shares.drop(columns=top).sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_df.plot(kind="barh", stacked=True, color=PALETA, width=0.65, ax=ax,
             edgecolor="white", linewidth=1.5)
ax.set_title(f"Composición de egresados de grado por campo de estudio, {anio}",
             loc="left", fontsize=12)
ax.set_xlabel("% de egresados (ED6)")
ax.set_ylabel("")
ax.set_xlim(0, 100)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.xaxis.grid(True, color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# Evolución del share de egresados TIC (F061) en el grado (ED6)
foco = ["ARG", "DEU", "ESP", "FRA", "ITA"]
ed6_all = panel[panel["isced_level"] == "ED6"]
tot = ed6_all.groupby(["iso3", "year"])["graduates"].sum()
ict = (ed6_all[ed6_all["iscedf_narrow"] == "F061"]
       .groupby(["iso3", "year"])["graduates"].sum())
share_ict = (ict / tot * 100).rename("share").reset_index()

fig, ax = plt.subplots(figsize=(9, 4.5))
for color, iso in zip(PALETA, foco):
    s = share_ict[share_ict["iso3"] == iso].sort_values("year")
    ax.plot(s["year"], s["share"], color=color, lw=2, label=iso)
    ax.annotate(iso, (s["year"].iloc[-1], s["share"].iloc[-1]),
                xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9, color="#444444")
ax.set_title("Egresados de TIC (F061) como % del total de grado",
             loc="left", fontsize=12)
ax.set_ylabel("%")
ax.set_ylim(bottom=0)
ax.legend(frameon=False, fontsize=8, loc="upper left")
for s_ in ("top", "right"):
    ax.spines[s_].set_visible(False)
ax.yaxis.grid(True, color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Salidas

- `data/processed/panel.parquet` — panel `iso3, year, isced_level, iscedf_narrow, graduates, source`
- `data/processed/coverage.csv` — cobertura narrow vs broad por país (Eurostat)

**Próximos pasos:** cruzar el panel con indicadores de desarrollo
(p. ej. World Bank WDI) por `iso3 × year`.